# U3-02 · Clasificación y agrupamiento de las judías secas

*Especialista en Inteligencia Artificial (IFCD107) · Unidad 3*

Ejecuta cada celda con **Mayús + Intro**. Todo funciona en el navegador; la primera vez tarda unos segundos en cargar las bibliotecas.

> Datos: Koklu, M. y Ozkan, I. A. (2020). *Computers and Electronics in Agriculture*, 174, 105507. https://doi.org/10.1016/j.compag.2020.105507 · Dry Bean, UCI, CC BY 4.0, https://doi.org/10.24432/C50S4B
> Datos: Fanaee-T, H. y Gama, J. (2014). Event labeling combining ensemble detectors and background knowledge. *Progress in Artificial Intelligence*, 2, 113–127. Bike Sharing, UCI, CC BY 4.0, https://doi.org/10.24432/C5W894

## 3.3 · Clasificación: la variedad de una judía

`i1_judias.py`

In [ ]:
import pandas as pd

j = pd.read_csv("judias.csv")
print(j.shape, "| clases:", j.Class.value_counts().to_dict())
print(j[["Area", "Perimeter", "roundness", "ShapeFactor2"]]
      .describe().loc[["mean", "min", "max"]].round(4).to_string())
c = j.drop(columns="Class").corr().abs()
pares = (c.where(~pd.DataFrame(
    [[i <= k for k in range(16)] for i in range(16)],
    index=c.index, columns=c.columns)).stack()
    .sort_values(ascending=False))
print("pares más correlacionados:")
print(pares.head(4).round(3).to_string())
print("duplicados:", j.duplicated().sum())
print(j.groupby("Class").Area.median().sort_values().astype(int)
      .to_string())

`i2_comparar.py`

In [ ]:
import time
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import (RandomForestClassifier,
                              HistGradientBoostingClassifier)
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold

j = pd.read_csv("judias.csv").drop_duplicates()
X, y = j.drop(columns="Class"), j.Class
modelos = {
    "regresión logística": LogisticRegression(max_iter=2000),
    "10 vecinos": KNeighborsClassifier(10),
    "Bayes ingenuo": GaussianNB(),
    "árbol (prof. 8)": DecisionTreeClassifier(max_depth=8,
                                              random_state=0),
    "SVM (RBF)": SVC(),
    "bosque aleatorio": RandomForestClassifier(300, n_jobs=-1,
                                               random_state=0),
    "potenciación": HistGradientBoostingClassifier(random_state=0),
    "red (64, 64)": MLPClassifier((64, 64), max_iter=500,
                                  random_state=0)}
cv = StratifiedKFold(5, shuffle=True, random_state=0)
for nombre, m in modelos.items():
    r = cross_validate(make_pipeline(StandardScaler(), m), X, y,
                       cv=cv, scoring=["accuracy", "f1_macro"])
    print(f"{nombre:<20} acierto {r['test_accuracy'].mean():.4f}"
          f"  F1 macro {r['test_f1_macro'].mean():.4f}"
          f"  ({r['fit_time'].sum():5.1f} s)")

`i5_escalar.py`

In [ ]:
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

j = pd.read_csv("judias.csv").drop_duplicates()
X, y = j.drop(columns="Class"), j.Class
print("desviación típica de Area:", round(X.Area.std()),
      "| de ShapeFactor2:", round(X.ShapeFactor2.std(), 5))
cv = StratifiedKFold(5, shuffle=True, random_state=0)
for k in [1, 5, 15, 50, 200]:
    sin = cross_val_score(KNeighborsClassifier(k), X, y, cv=cv)
    con = cross_val_score(make_pipeline(
        StandardScaler(), KNeighborsClassifier(k)), X, y, cv=cv)
    print(f"k={k:3d}: sin escalar {sin.mean():.4f}"
          f"  escalando {con.mean():.4f}")

`i3_matriz.py`

In [ ]:
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

j = pd.read_csv("judias.csv").drop_duplicates()
X, y = j.drop(columns="Class"), j.Class
X_e, X_p, y_e, y_p = train_test_split(X, y, test_size=0.25,
                                      stratify=y, random_state=0)
m = make_pipeline(StandardScaler(), SVC()).fit(X_e, y_e)
p = m.predict(X_p)
cl = sorted(y.unique())
abrev = [c[:4] for c in cl]
cm = pd.DataFrame(confusion_matrix(y_p, p, labels=cl),
                  index=abrev, columns=abrev)
print(cm.to_string())
print(classification_report(y_p, p, digits=3))

`i6_umbral.py`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, precision_score,
                             recall_score)

j = pd.read_csv("judias.csv").drop_duplicates()
j = j[j.Class.isin(["SIRA", "DERMASON"])]
X, y = j.drop(columns="Class"), (j.Class == "SIRA").astype(int)
X_e, X_p, y_e, y_p = train_test_split(X, y, test_size=0.25,
                                      stratify=y, random_state=0)
m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
prob = m.fit(X_e, y_e).predict_proba(X_p)[:, 1]   # P(Sira)
print(f"AUC: {roc_auc_score(y_p, prob):.3f}")
for u in [0.2, 0.5, 0.8]:
    p = (prob >= u).astype(int)
    print(f"umbral {u}: precisión {precision_score(y_p, p):.3f}"
          f"  sensibilidad {recall_score(y_p, p):.3f}"
          f"  marcadas Sira {p.sum()}")
dudosas = ((prob > 0.2) & (prob < 0.8)).mean()
print(f"probabilidad entre 0,2 y 0,8: {dudosas:.1%} de las judías")

`i4_svm.py`

In [ ]:
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

j = pd.read_csv("judias.csv").drop_duplicates()
j = j[j.Class.isin(["SIRA", "DERMASON"])]   # el par más difícil
X, y = j.drop(columns="Class"), j.Class
X_e, X_p, y_e, y_p = train_test_split(X, y, test_size=0.25,
                                      stratify=y, random_state=0)
for nucleo in ["linear", "rbf"]:
    for C in [0.01, 1, 100]:
        m = make_pipeline(StandardScaler(), SVC(kernel=nucleo, C=C))
        m.fit(X_e, y_e)
        print(f"{nucleo:<6} C={C:<5} vectores soporte "
              f"{m[-1].n_support_.sum():4d}/{len(y_e)}"
              f"  entr. {m.score(X_e, y_e):.3f}"
              f"  prueba {m.score(X_p, y_p):.3f}")

`i7_reglas.py`

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import cross_val_score, StratifiedKFold

j = pd.read_csv("judias.csv").drop_duplicates()
X, y = j.drop(columns="Class"), j.Class
cv = StratifiedKFold(5, shuffle=True, random_state=0)
for prof in [1, 2, 3, 4, 6]:
    a = DecisionTreeClassifier(max_depth=prof, random_state=0)
    print(f"profundidad {prof}: acierto CV "
          f"{cross_val_score(a, X, y, cv=cv).mean():.3f}")
a = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X, y)
print(export_text(a, feature_names=list(X.columns), decimals=4))

## 3.4 · Aprendizaje no supervisado

`j1_sin_etiquetas.py`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

j = pd.read_csv("judias.csv").drop_duplicates()
Z = StandardScaler().fit_transform(j.drop(columns="Class"))
pca = PCA().fit(Z)
print("varianza explicada acumulada:",
      pca.explained_variance_ratio_.cumsum()[:4].round(3))
P = pca.transform(Z)[:, :2]
for k in [3, 7, 10]:
    g = KMeans(k, n_init=10, random_state=0).fit_predict(Z)
    print(f"k-medias k={k:2d}: ARI con la variedad "
          f"{adjusted_rand_score(j.Class, g):.3f}")
g = KMeans(7, n_init=10, random_state=0).fit_predict(Z)
print(pd.crosstab(g, j.Class.str[:4], rownames=["grupo"],
                  colnames=[""]).to_string())
fig, eje = plt.subplots(figsize=(4.4, 2.8))
for n, (c, d) in enumerate(pd.DataFrame(P).groupby(
        j.Class.to_numpy())):
    eje.scatter(d[0], d[1], s=0.5, alpha=0.4, label=c.title(),
                color=plt.cm.tab10(n))
eje.set_xlabel("CP1"); eje.set_ylabel("CP2")
eje.legend(markerscale=8, fontsize=6); plt.show()

`j2_jerarquico.py`

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score

j = pd.read_csv("judias.csv").drop_duplicates()
Z = StandardScaler().fit_transform(j.drop(columns="Class"))
sil = lambda g: silhouette_score(Z, g, sample_size=4000,
                                 random_state=0)
print("k  silueta k-medias  silueta Ward  ARI Ward")
for k in [2, 3, 4, 5, 6, 7, 8]:
    km = KMeans(k, n_init=10, random_state=0).fit_predict(Z)
    wa = AgglomerativeClustering(k, linkage="ward").fit_predict(Z)
    print(f"{k}  {sil(km):16.3f}  {sil(wa):12.3f}"
          f"  {adjusted_rand_score(j.Class, wa):8.3f}")

`j3_anomalias.py`

In [ ]:
import pandas as pd
from sklearn.ensemble import IsolationForest
from bicis import X, h

d = pd.DataFrame({"fecha": h.dteday, "hora": h.hr, "n": h.cnt,
                  "temp_c": X.temp_c, "humedad": X.humedad})
dia = d.groupby("fecha").agg(alquileres=("n", "sum"),
                             horas=("hora", "size"),
                             temp=("temp_c", "mean"),
                             humedad=("humedad", "mean"))
dia["laborable"] = h.groupby("dteday").workingday.first()
iso = IsolationForest(n_estimators=300, contamination=0.01,
                      random_state=0).fit(dia)
marca = iso.predict(dia) == -1
dia["rareza"] = -iso.score_samples(dia)
print(dia.sort_values("rareza", ascending=False).head(8)
      .round(2).to_string())
print("días marcados:", marca.sum(), "de", len(dia))

**Prueba.** En `i3_matriz.py`, sustituye la SVM por la potenciación. ¿Cambian los pares de variedades que se confunden?